In [34]:
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util as qz
import ifcopenshell.util.element as qx
model = ifcopenshell.open("Archicad-Zone.ifc")

# model = ifcopenshell.open("AC20-FZK-Haus.ifc")


# declare wall objects IfcWall
walls = model.by_type('IfcWall')
print(walls)


# wall = model.by_type('IfcWall')[0]
# print(wall.id())


# def get_opening_list_from_wall(wall):
#     """
#     Get the list of openings associated with the given wall.
 
#     :param wall: The wall object.
#     :type wall: IfcWall
 
#     :return: List of IfcOpening objects associated with the wall.
#     :rtype: list
#     """
#     opening_list = wall.HasOpenings
#     return opening_list

[#211=IfcWallStandardCase('100_BUDOj7ThPb2CKAun_F',#2,'Wall-023',$,$,#383,#99,'4003E2DE-358B-4776-B665-08C50AE31F8F'), #243=IfcWallStandardCase('3ngTTjQ3X3teOuK9AzlHx8',#2,'Wall-021',$,$,#488,#119,'F1A9D76D-6838-43DE-8638-5092BDBD1EC8'), #248=IfcWallStandardCase('2uO6m2yCT0ju7FBMSmIsDA',#2,'Wall-022',$,$,#511,#123,'B8606C02-F0C7-40B7-81CF-2D67304B634A'), #252=IfcWallStandardCase('00YQDNiwb97Bg0Rt8t8AFC',#2,'Wall-025',$,$,#531,#125,'0089A357-B3A9-491C-BA80-6F723720A3CC'), #256=IfcWallStandardCase('3OUhFO8jn2IfZ_wR$Zhjmx',#2,'Wall-028',$,$,#550,#127,'D87AB3D8-22DC-424A-98FE-E9BFE3AEDC3B'), #260=IfcWallStandardCase('2wFROZHfr719s0K0YmoEJQ',#2,'Wall-029',$,$,#570,#129,'BA3DB623-469D-4704-9D80-5008B0C8E4DA'), #264=IfcWallStandardCase('2BxgiXGhj9FON96wzMzXeg',#2,'Wall-037',$,$,#590,#131,'8BEEAB21-42BB-493D-85C9-1BAF56F61A2A'), #268=IfcWallStandardCase('1VAwblXYXFuODB_1ZRqZEa',#2,'Wall-038',$,$,#610,#133,'5F2BA96F-8628-4FE1-834B-F818DBD233A4')]


In [27]:
# number of walls. For this model, "walls" only has 4 elements, which makes it easy
no_walls = len(walls)
print('Number of walls is ', no_walls)

print('Properties are ', walls[0])

wall = walls[0]

print('Loop start--------')
for i in range(0,len(wall)):
    print('Type is ',type(wall[i]),'. Value is ', wall[i])


# this is just a way to get direct property sets from the Ifcwall. Likely need to look at inverse relationship formulas
# ifcopenshell.entity_instance.entity_instance

Number of walls is  8
Properties are  #297=IfcWall('100_BUDOj7ThPb2CKAun_F',#2,'Wall-023',$,$,#2089,#112,'4003E2DE-358B-4776-B665-08C50AE31F8F',.NOTDEFINED.)
Loop start--------
Type is  <class 'str'> . Value is  100_BUDOj7ThPb2CKAun_F
Type is  <class 'ifcopenshell.entity_instance.entity_instance'> . Value is  #2=IfcOwnerHistory(#6,#7,$,.NOCHANGE.,$,$,$,1776688393)
Type is  <class 'str'> . Value is  Wall-023
Type is  <class 'NoneType'> . Value is  None
Type is  <class 'NoneType'> . Value is  None
Type is  <class 'ifcopenshell.entity_instance.entity_instance'> . Value is  #2089=IfcLocalPlacement(#2077,#2170)
Type is  <class 'ifcopenshell.entity_instance.entity_instance'> . Value is  #112=IfcProductDefinitionShape($,$,(#36,#77))
Type is  <class 'str'> . Value is  4003E2DE-358B-4776-B665-08C50AE31F8F
Type is  <class 'str'> . Value is  NOTDEFINED


In [28]:
# check which walls have openings
for i in range(len(walls)):
    wallOpen = walls[i].HasOpenings
    if (wallOpen):
        print('Wall ',i,' has ', len(wallOpen),' openings')

Wall  0  has  2  openings
Wall  7  has  1  openings


In [29]:
# example obtaining opening dimensions from a single wall
for rel in walls[0].HasOpenings:
    opening = rel.RelatedOpeningElement
    print("Name is ", opening.Name)
    
    # this get property sets for the specific opening
    psets = qx.get_psets(opening)

    # This determines the exact name of the "proprties" used. Sometimes it could be pset, or qto when finding window dimensions
    # print(psets.keys())

    # since the dimensions could be stored in two different properties, we declare variables that can compute both i.e. one will be legit, the other will be empty
    qto = psets.get("Qto_OpeningElementBaseQuantities",{})
    pset_common = psets.get('Pset_OpeningElementCommon',{})

    width = qto.get("Width") or pset_common.get("Width")
    height = qto.get("Height") or pset_common.get("Height")
    depth = qto.get("Depth") or pset_common.get("Depth")
    print("WxHxD = ",width,'x',height,'x',depth, '\n')

Name is  D-007
WxHxD =  None x None x None 

Name is  D-009
WxHxD =  None x None x None 



For my next trick, I'm going to try to obtain the wall property materials

In [30]:
for wall in walls:
    material = qx.get_material(wall)

    # first, work out the number of layers, and the materials used
    if (material.is_a("IfcMaterialLayerSetUsage")):
        layer_set = material.ForLayerSet
        print('Number of Wall Layers is ',len(layer_set.MaterialLayers))

        for layer in layer_set.MaterialLayers:
            # obtain material name
            mat_name = layer.Material.Name
            thickness = layer.LayerThickness
            print('Material is ',mat_name,'\n Thickness is ',thickness,'\n')


            # obtain thermal properties of the wall layer
            # https://standards.buildingsmart.org/IFC/RELEASE/IFC2x3/TC1/HTML/ifcmaterialpropertyresource/lexical/ifcthermalmaterialproperties.htm
            thermal_pset = qx.get_psets(layer.Material).get("Pset_MaterialThermal",{})
            conductivity = thermal_pset.get("ThermalConductivity")
            specific_heat = thermal_pset.get("SpecificHeatCapacity")
            # print(specific_heat)

In [31]:
for wall in walls:
    material = qx.get_material(wall)

Aim:
To extract information from Ifc models using IfcOpenShell for use in designing the Fire Safety strategy

Objectives:
- Obtain property materials from a room, and use it to develop a design fire
- Obtain compartment size dimensions and opening, to calculate flashover conditions during a compartment fire
- Automatically compute travel distances within a building

OR...

Aim:
To evaluate travel distances within a building from Ifc models using IfcOpenShell

Objectives:
- Construct rooms within a building using nodes and graphs
- Compute travel distance between nodes using shortest straight line path
- Find longest paths of travel (this looks hard)
- Validate travel distance using pathfinder

https://en.wikipedia.org/wiki/Price%27s_model


In [32]:
# Extract IFC Spaces
spaces = model.by_type('IfcSpace')
print(f"Number of spaces: {len(spaces)}")

for space in spaces:
    print(f"Space ID: {space.id()}, Name: {space.Name}")
    
    # Get geometry and position
    try:
        # Create geometry settings
        settings = ifcopenshell.geom.settings()
        settings.set(settings.USE_WORLD_COORDS, True)
        
        # Get the shape
        shape = ifcopenshell.geom.create_shape(settings, space)
        
        # Get vertices (geometry data)
        verts = shape.geometry.verts
        print(f"Space {space.Name} has {len(verts)//3} vertices")
        
        # For position, you can get the placement
        placement = space.ObjectPlacement
        if placement:
            if placement.is_a('IfcLocalPlacement'):
                # Get the relative placement
                rel_placement = placement.RelativePlacement
                if rel_placement and rel_placement.is_a('IfcAxis2Placement3D'):
                    location = rel_placement.Location
                    print(f"Position: ({location.Coordinates[0]}, {location.Coordinates[1]}, {location.Coordinates[2]})")
                    
    except Exception as e:
        print(f"Error getting geometry for space {space.Name}: {e}")

Number of spaces: 0
